# AGMTL-DenseCBAM — Leakage-Safe Training

This notebook is a thin interface to the active Python pipeline. Keeping model and evaluation logic in one implementation prevents the notebook and scripts from drifting apart.

Final workflow: audit data → generate leakage-safe folds → smoke test → run C1–C4 → aggregate Chapter IV results.

In [ ]:
# Run once in a clean Colab/runtime. TensorFlow may already be installed.
# %pip install -r requirements.txt

import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'data'
FOLDS_ROOT = PROJECT_ROOT / 'data_5_fold'
RESULTS_ROOT = PROJECT_ROOT / 'chapter4_results'
print('Python:', sys.version)
print('Project:', PROJECT_ROOT)

## 1. Audit and generate folds

The default `error` policy writes `duplicate_audit.csv` and stops when identical pixels have conflicting labels. Review those rows with the dataset owner. Change to `exclude` only when exclusion has been formally approved and will be documented in the thesis. Add `--groups_csv ...` when patient/study IDs are available.

In [ ]:
CONFLICT_POLICY = 'error'  # change to 'exclude' only after review/approval
fold_command = [
    sys.executable, 'make_5fold_dataset.py',
    '--input_root', str(DATA_ROOT),
    '--output_root', str(FOLDS_ROOT),
    '--n_splits', '5', '--val_size', '0.15', '--seed', '42',
    '--conflict_policy', CONFLICT_POLICY,
]
subprocess.run(fold_command, check=True)

## 2. Smoke test

This uses one fold and two epochs in a separate output directory. It is not a reportable result.

In [ ]:
smoke_command = [
    sys.executable, 'run_case_5fold_isolated.py',
    '--folds_root', str(FOLDS_ROOT),
    '--model_type', 'proposed', '--scenario', 'artifact_mix',
    '--epochs', '2', '--fold_limit', '1',
    '--output_dir', str(RESULTS_ROOT / 'smoke_proposed_artifact_mix'),
]
subprocess.run(smoke_command, check=True)

## 3. Final C1–C4 runs

Run all cases on the same folds, seed, hyperparameters, and hardware. A fresh process is used for each fold. This can take many hours.

In [ ]:
EPOCHS = 50
CASES = [
    ('benchmark', 'clean'),
    ('benchmark', 'artifact_mix'),
    ('proposed', 'clean'),
    ('proposed', 'artifact_mix'),
]

for model_type, scenario in CASES:
    output_dir = RESULTS_ROOT / f'{model_type}_{scenario}'
    command = [
        sys.executable, 'run_case_5fold_isolated.py',
        '--folds_root', str(FOLDS_ROOT),
        '--model_type', model_type, '--scenario', scenario,
        '--epochs', str(EPOCHS), '--batch_size', '8',
        '--learning_rate', '1e-4', '--l2_strength', '1e-2',
        '--seed', '42', '--output_dir', str(output_dir),
    ]
    print('\nRUNNING:', model_type, scenario)
    subprocess.run(command, check=True)

## 4. Chapter IV aggregate analysis

The analyzer refuses incomplete case sets. Outputs include mean ± SD, 95% CIs, paired tests, robustness degradation comparisons, and graphs.

In [ ]:
subprocess.run([
    sys.executable, 'analyze_chapter4.py',
    '--results_root', str(RESULTS_ROOT),
    '--expected_folds', '5',
], check=True)

## Interpretation safeguards

- Do not report `oldstyle_results` or smoke-test metrics as final results.
- State whether patient/study grouping was available. Hash deduplication alone does not establish patient independence.
- Describe artifacts as synthetic corruptions, not verified real clinical artifact classes.
- Five-fold inferential tests have low power; report fold values, confidence intervals, and effect sizes with p-values.
- Grad-CAM remains qualitative unless expert ROI annotations are available.